In [6]:
from pathlib import Path

GTFS_PATH = Path.cwd().parent / "data" / "raw" / "gtfs_vmt" / "extracted"

print(f"GTFS-Dateien von {GTFS_PATH}")
for file in GTFS_PATH.glob("*.txt"):
    print(f"{file.name}")

GTFS-Dateien von /Users/leon/Projects/transIT/data/raw/gtfs_vmt/extracted
transfers.txt
agency.txt
calendar_dates.txt
stop_times.txt
frequencies.txt
shapes.txt
trips.txt
feed_info.txt
stops.txt
calendar.txt
routes.txt


In [ ]:
#Wichtigsten Dateien inkl. Dimensionen laden
import pandas as pd

routes=pd.read_csv(GTFS_PATH/"routes.txt") #-->  Linien (Route=Linie)
trips=pd.read_csv(GTFS_PATH/"trips.txt") #--> Linienrouten (trip=FAhrt)
stops=pd.read_csv(GTFS_PATH/"stops.txt") #--> Hst
agency=pd.read_csv(GTFS_PATH/"agency.txt") #--> VU
stop_times = pd.read_csv(GTFS_PATH/"stop_times.txt") #-->Haltezeiten

print(f"Routes {routes.shape}")
print(f"{routes.dtypes} \n")

print(f"Trips {trips.shape}")
print(f"{trips.dtypes} \n")

print(f"Stops {stops.shape}")
print(f"{stops.dtypes} \n")

print(f"Agency {agency.shape}")
print(f"{agency.dtypes} \n")

print(f"Stop Times {stop_times.shape}")
print(f"{stop_times.dtypes} \n")



"""
display(routes.head())
display(trips.head())
display(stops.head())
display(agency.head())
display(stop_times())
"""


Routes (857, 8)
route_id                str
agency_id             int64
route_short_name        str
route_long_name     float64
route_type            int64
route_color             str
route_text_color        str
route_desc          float64
dtype: object 

Trips (115512, 10)
route_id                     str
service_id                 int64
trip_id                    int64
trip_headsign                str
trip_short_name          float64
direction_id               int64
block_id                 float64
shape_id                 float64
wheelchair_accessible      int64
bikes_allowed              int64
dtype: object 

Stops (13169, 10)
stop_id                    str
stop_code              float64
stop_name                  str
stop_desc              float64
stop_lat               float64
stop_lon               float64
location_type            int64
parent_station         float64
wheelchair_boarding      int64
platform_code              str
dtype: object 

Agency (44, 6)
agency_id           

/var/folders/g5/115sqwx15x7cmsl32yg56d_80000gn/T/ipykernel_1533/2094997739.py:8: DtypeWarning: Columns (0: stop_headsign) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(GTFS_PATH/"stop_times.txt") #-->


'\ndisplay(routes.head())\ndisplay(trips.head())\ndisplay(stops.head())\ndisplay(agency.head())\n'

In [26]:
#Alle VU in ganz Thüringen
agency[["agency_id", "agency_name"]].sort_values("agency_name")

,agency_id,agency_name
29,176,Abellio Rail Mitteldeutschland GmbH
41,237,Busbetrieb Piehler GmbH & Co. KG
4,78,Deutsche Bahn
21,131,EW Bus GmbH
28,172,Erfurter Bahn
27,171,Erfurter Bahn
16,119,Erfurter Verkehrsbetriebe AG (EVAG)
34,211,Firma Schieck
9,111,GVB Verkehrs- und Betriebsgesellschaft Gera mb...
26,166,Harzer Schmalspurbahnen GmbH


In [27]:
#Merkmale ausgeben
print("Routes:", routes.columns.tolist())
print("Trips:", trips.columns.tolist())
print("Stops:", stops.columns.tolist())
print("Agency:", agency.columns.tolist())
print("Stop times:", stop_times.columns.tolist())

Routes: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'route_desc']
Trips: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'bikes_allowed']
Stops: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code']
Agency: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']
Stop times: ['trip_id', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'stop_headsign', 'arrival_time', 'departure_time']


In [ ]:
#Schlüsselkonsistenzen

print("Agency IDs:", agency["agency_id"].nunique(), "/", len(agency))
print("Route IDs:", routes["route_id"].nunique(), "/", len(routes))
print("Trip IDs:", trips["trip_id"].nunique(), "/", len(trips))
print("Stop IDs:", stops["stop_id"].nunique(), "/", len(stops))

Agency IDs: 44 / 44
Route IDs: 857 / 857
Trip IDs: 115512 / 115512
Stop IDs: 13169 / 13169


In [29]:

display(stop_times.head())

,trip_id,stop_id,stop_sequence,pickup_type,drop_off_type,stop_headsign,arrival_time,departure_time
0,40063150,de:16063:163004::16300402,0,0,0,NaN,9:30:00,9:30:00
1,40063150,de:16063:163013::16301300,1,0,0,NaN,9:32:00,9:32:00
2,40063150,de:16063:163005::16300500,2,0,0,NaN,9:33:00,9:33:00
3,40063150,de:16063:163011::16301100,3,0,0,NaN,9:35:00,9:35:00
4,40063150,de:16063:1700754::170075400,4,0,0,NaN,9:38:00,9:38:00


In [30]:
#Test auf Foreignkeyabhängigkeiten

#routes -> agency
invalid_route_agencies = routes[~routes["agency_id"].isin(agency["agency_id"])]
print("Routes mit unbekannter agency_id:",len(invalid_route_agencies))
print("Unbekannte trip_id:",(~stop_times["trip_id"].isin(trips["trip_id"])).sum())
print("Unbekannte stop_id:",(~stop_times["stop_id"].isin(stops["stop_id"])).sum())

#trips -> routes
invalid_trip_routes = trips[~trips["route_id"].isin(routes["route_id"])]
print("Trips mit unbekannter route_id:",len(invalid_trip_routes))

#stop_times -> stops
print("Unbekannte trip_id:",(~stop_times["trip_id"].isin(trips["trip_id"])).sum())
print("Unbekannte stop_id:",(~stop_times["stop_id"].isin(stops["stop_id"])).sum())

Routes mit unbekannter agency_id: 0
Unbekannte trip_id: 0
Unbekannte stop_id: 0
Trips mit unbekannter route_id: 0
Unbekannte trip_id: 0
Unbekannte stop_id: 0


In [ ]:
#Zeitdaten laden

#Fahrplankalender
calendar = pd.read_csv(GTFS_PATH/"calendar.txt") #1=Ja, 0=Nein
#Calendardates: Ausnahmen des Fahrplankalenders --> 1: Zusätzliche Fahrt und 2: Fahrt entfernen
calendar_dates = pd.read_csv(GTFS_PATH/"calendar_dates.txt") #1=Zusätzliche Fahrt

print(f"Calendar {calendar.shape}")
print(f"{calendar.dtypes} \n")

print(f"Calendar dates {calendar_dates.shape}")
print(f"{calendar_dates.dtypes} \n")


print("calendar:", calendar.columns.tolist())
print("Calendar dates", calendar_dates.columns.tolist()) #Service-id: Verbindung zwischen PlanFahrt und Fahrplankalender

Calendar (2273, 10)
service_id    int64
monday        int64
tuesday       int64
wednesday     int64
thursday      int64
friday        int64
saturday      int64
sunday        int64
start_date    int64
end_date      int64
dtype: object 

Calendar dates (67385, 3)
service_id        int64
date              int64
exception_type    int64
dtype: object 

calendar: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Calendar dates ['service_id', 'date', 'exception_type']


In [36]:
print("Calendar:")
print("Start:", calendar["start_date"].min(), "End:", calendar["end_date"].max())

print(f"\ncalendar_dates:")
print("Datum:",calendar_dates["date"].min(), "bis", calendar_dates["date"].max())

print(f"\nAnzahl service_id:")
print("calendar:", calendar["service_id"].nunique())
print("calendar_dates:", calendar_dates["service_id"].nunique())

Calendar:
Start: 20260801 End: 20270228

calendar_dates:
Datum: 20260801 bis 20270228

Anzahl service_id:
calendar: 2273
calendar_dates: 2266
